In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/pet-breed-classifier

import torch
print(torch.__version__, '| GPU:', torch.cuda.is_available())

Mounted at /content/drive
/content/drive/MyDrive/pet-breed-classifier
2.11.0+cu130 | GPU: True


In [ ]:
%%writefile -a src/models.py


class MediumCNN(nn.Module):
    """SimpleCNN-ზე უფრო დიდი: 5 conv ბლოკი (512 არხამდე) + 2-ფენიანი კლასიფიკატორი."""

    def __init__(self, num_classes=37, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),     # 224 -> 112
            conv_block(32, 64),    # 112 -> 56
            conv_block(64, 128),   # 56  -> 28
            conv_block(128, 256),  # 28  -> 14
            conv_block(256, 512),  # 14  -> 7
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

Appending to src/models.py


In [ ]:
import torch
from src.models import MediumCNN

model = MediumCNN(num_classes=37)
out = model(torch.randn(4, 3, 224, 224))

print('გამოსასვლელის ფორმა:', out.shape)
print('პარამეტრები:', sum(p.numel() for p in model.parameters()))

გამოსასვლელის ფორმა: torch.Size([4, 37])
პარამეტრები: 1711397


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/pet-breed-classifier

import torch
print(torch.__version__, '| GPU:', torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/pet-breed-classifier
2.11.0+cu130 | GPU: True


In [ ]:
!grep -n "^def \|^import \|^class " src/train.py
!wc -l src/train.py

1:import torch
2:import torch.nn as nn
5:def train_one_epoch(model, loader, criterion, optimizer, device):
30:def evaluate(model, loader, criterion, device):
59:import time
62:def fit(model, train_loader, val_loader, epochs, lr, device, ckpt_path,
95 src/train.py


In [ ]:
%%writefile -a src/train.py

import time


def fit(model, train_loader, val_loader, epochs, lr, device, ckpt_path,
        weight_decay=1e-4, label_smoothing=0.0, use_wandb=False):
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc]):
            history[k].append(v)

        if use_wandb:
            import wandb
            wandb.log({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
                       "val_loss": va_loss, "val_acc": va_acc,
                       "lr": optimizer.param_groups[0]["lr"]})

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), ckpt_path)

        print(f"epoch {epoch:02d}/{epochs} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
              f"val loss {va_loss:.3f} acc {va_acc:.3f} | {time.time() - t0:.0f}s")

    print(f"საუკეთესო val acc: {best_val_acc:.3f}")
    return history

Appending to src/train.py


In [ ]:
import torch, json
from src.data import get_dataloaders
from src.models import MediumCNN
from src.train import fit

device = 'cuda'
torch.manual_seed(42)

train_loader, val_loader, test_loader, classes = get_dataloaders(
    root='/content/data', batch_size=32, num_workers=0)

model = MediumCNN(num_classes=37, dropout=0.4)

history = fit(model, train_loader, val_loader,
              epochs=15, lr=1e-3, device=device,
              ckpt_path='models/medium_cnn.pth')

with open('reports/medium_history.json', 'w') as f:
    json.dump(history, f)

epoch 01/15 | train loss 3.570 acc 0.055 | val loss 3.533 acc 0.052 | 32s
epoch 02/15 | train loss 3.457 acc 0.070 | val loss 3.598 acc 0.050 | 33s
epoch 03/15 | train loss 3.394 acc 0.087 | val loss 3.415 acc 0.083 | 32s
epoch 04/15 | train loss 3.339 acc 0.095 | val loss 3.465 acc 0.071 | 32s
epoch 05/15 | train loss 3.252 acc 0.111 | val loss 3.428 acc 0.090 | 32s
epoch 06/15 | train loss 3.156 acc 0.127 | val loss 3.383 acc 0.091 | 32s
epoch 07/15 | train loss 3.050 acc 0.155 | val loss 3.284 acc 0.109 | 32s
epoch 08/15 | train loss 2.971 acc 0.175 | val loss 3.161 acc 0.136 | 33s
epoch 09/15 | train loss 2.897 acc 0.184 | val loss 3.006 acc 0.170 | 32s
epoch 10/15 | train loss 2.823 acc 0.191 | val loss 2.927 acc 0.193 | 32s
epoch 11/15 | train loss 2.762 acc 0.210 | val loss 2.942 acc 0.192 | 32s
epoch 12/15 | train loss 2.698 acc 0.233 | val loss 2.819 acc 0.202 | 32s
epoch 13/15 | train loss 2.656 acc 0.243 | val loss 2.780 acc 0.230 | 32s
epoch 14/15 | train loss 2.605 acc 0.2

In [ ]:
!sed -n '25,60p' src/train.py


    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    """ვალიდაციის/ტესტირების ფუნქცია (Validation Loop)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / total
    val_acc = correct / total
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    return val_loss, val_acc, val_f1

im

In [ ]:
p = 'src/train.py'
s = open(p).read()
s = s.replace(
    "va_loss, va_acc = evaluate(model, val_loader, criterion, device)",
    "va_loss, va_acc, va_f1 = evaluate(model, val_loader, criterion, device)")
open(p, 'w').write(s)

!grep -n "f1_score\|va_f1" src/train.py

3:from sklearn.metrics import f1_score
56:    val_f1 = f1_score(all_labels, all_preds, average='macro')
75:        va_loss, va_acc, va_f1 = evaluate(model, val_loader, criterion, device)
